# 06 — Gaussian predictor

`u_{n+1} ~ N(mu(u_n), Sigma(u_n))`, trained by maximum likelihood.

If sigma were fixed, minimising the likelihood would be exactly minimising MSE — so any gain
has to come from the sigma head and from sampling. The mean-vs-sampled test below is what
isolates that, and it now runs on all five seeds, because it is read on `alive` and `climate`.

`chaos` is absent from the control on purpose: the estimator switches sampling off itself, so
the two rows would carry the identical number by construction.

In [1]:
import sys, json, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks'
                       else pathlib.Path.cwd()))
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt, torch
from l63 import ARTIFACTS, evaluate as E, plots as P
from l63.data import make_datasets
from run.report import clean, load_model, load_rows, summarise

gt = json.load(open(ARTIFACTS / 'ground_truth.json'))
S  = clean(summarise(load_rows()))
DATA = list(gt['datasets'])                      # 'ode', 'sde', 'sde015'

def num(v, w=6, p=2):
    """A ruler value, or an em dash where it is genuinely undefined."""
    if isinstance(v, dict):
        v = v.get('median')
    undefined = v is None or v != v          # None from clean(), bare NaN from the json
    return f'{"—":>{w}}' if undefined else f'{v:{w}.{p}f}'

def rng(v, p=2):
    if v is None or v.get('lo') is None or v['lo'] != v['lo']:
        return '—'
    return f"[{v['lo']:.{p}f}–{v['hi']:.{p}f}]"

print(len(S), 'model x dataset entries ·', len(DATA), 'datasets')

43 model x dataset entries · 3 datasets


## Numbers

In [2]:
for key in DATA:
    s = S.get(f'06_gaussian_{key}')
    if s is None: continue
    print(f"--- {key.upper()} ---")
    for tag, r in (('sampled', s), ('mean only', s['mean_only'])):
        print(f"  {tag:10s} horizon {num(r['horizon'],4,0)}   climate {num(r['climate'])}   "
              f"spread {num(r['spread'])}   alive {num(r['alive'])}   {rng(r['alive'])}")
    sr = s.get('sigma_over_residual')
    if sr and sr.get('median'):
        print(f"  learned sigma is {sr['median']:.2f}x the model's own one-step error "
              f"{rng(sr)}")

--- ODE ---
  sampled    horizon  240   climate  18.58   spread      —   alive   1.00   [1.00–1.00]
  mean only  horizon  265   climate  17.12   spread      —   alive   1.00   [0.00–1.00]
  learned sigma is 7.75x the model's own one-step error [1.32–16.98]
--- SDE ---
  sampled    horizon   27   climate   5.18   spread   1.00   alive   1.00   [1.00–1.00]
  mean only  horizon   41   climate  28.56   spread   0.00   alive   1.00   [1.00–1.00]
  learned sigma is 0.90x the model's own one-step error [0.89–0.91]
--- SDE015 ---
  sampled    horizon   89   climate   6.50   spread   0.94   alive   1.00   [1.00–1.00]
  mean only  horizon  113   climate   9.02   spread   0.00   alive   1.00   [1.00–1.00]
  learned sigma is 0.97x the model's own one-step error [0.95–1.13]


## This model

In [3]:
KEY = 'ode'      # any of DATA
s = S['06_gaussian_' + KEY]
ref = gt['datasets'][KEY]
d = make_datasets(seed=0, kind=ref['kind'], b=ref['b'])
m, hist = load_model('06_gaussian_' + KEY + '_s' + str(s['rep_seed']))

print(f"{m.n_params:,} parameters, history {m.history}, figures show seed {s['rep_seed']}")
print()
print(f"{'ruler':16s}{'median':>8s}   range over seeds")
for k in ('horizon', 'spread', 'climate', 'climate_vs_truth', 'chaos', 'alive', 'lobe'):
    v = s[k]
    p = 0 if k == 'horizon' else 2
    print(f"  {k:14s}{num(v, 8, p)}   {rng(v, p)} over {v['n']} seeds")
print(f"\ntruth on this dataset:  climate {ref['truth_climate']:.2f}   "
      f"alive {ref['truth_alive']:.2f}   lobe {ref['truth_lobe']:.2f}   "
      f"ground truth usable {ref['floor_steps']} steps")
print(f"\nfirst steps, ||u_hat_n - u_n|| in Lorenz units:")
for i, e in enumerate(s['early'][:6], 1):
    print(f"  n={i}  {e:.3e}")

17,798 parameters, history 1, figures show seed 4

ruler             median   range over seeds
  horizon            240   [122–272] over 5 seeds
  spread               —   — over 0 seeds
  climate          18.58   [5.13–52.62] over 5 seeds
  climate_vs_truth   29.94   [8.26–84.80] over 5 seeds
  chaos             0.94   [0.13–1.05] over 5 seeds
  alive             1.00   [1.00–1.00] over 5 seeds
  lobe              0.63   [0.34–0.76] over 5 seeds

truth on this dataset:  climate 0.62   alive 1.00   lobe 0.63   ground truth usable 362 steps

first steps, ||u_hat_n - u_n|| in Lorenz units:
  n=1  9.882e-02
  n=2  1.128e-01
  n=3  1.406e-01
  n=4  1.599e-01
  n=5  1.876e-01
  n=6  2.108e-01


## Figures

Banked by `run/report.py`; regenerated here from the same checkpoint so the notebook and the deck cannot disagree.

In [4]:
P.loss_figure(hist, None, n_val_traj=8); plt.show()
P.arch_figure(m.spec(), "", m.n_params, None); plt.show()
long = d.raw(m.forecast(d.eval[:gt['n_long'], :m.history], gt['long_steps']))
P.lorenz_map_figure(long, d.raw(d.eval), None); plt.show()

/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50622/4260566466.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.loss_figure(hist, None, n_val_traj=8); plt.show()
/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50622/4260566466.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.arch_figure(m.spec(), "", m.n_params, None); plt.show()


/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50622/4260566466.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.lorenz_map_figure(long, d.raw(d.eval), None); plt.show()


## Findings

_Written after reading the numbers above._

- 
- 
- 